In [5]:
import pandas as pd
import numpy as np

# Load all three files
products = pd.read_csv('../data/product_master.csv')
stock = pd.read_csv('../data/branch_stock.csv')
usage = pd.read_csv('../data/weekly_usage.csv')

print("=== DATA VALIDATION ===")
print(f"Products: {products.shape} | Nulls: {products.isnull().sum().sum()}")
print(f"Stock: {stock.shape} | Nulls: {stock.isnull().sum().sum()}")
print(f"Usage: {usage.shape} | Nulls: {usage.isnull().sum().sum()}")
print(f"\nBranches: {stock['Branch'].unique()}")
print(f"Categories: {products['Category'].unique()}")
print(f"Weeks covered: {usage['Week'].min()} to {usage['Week'].max()}")

=== DATA VALIDATION ===
Products: (20, 10) | Nulls: 0
Stock: (80, 8) | Nulls: 0
Usage: (960, 6) | Nulls: 0

Branches: <StringArray>
['North Branch', 'South Branch', 'East Branch', 'West Branch']
Length: 4, dtype: str
Categories: <StringArray>
['PPE', 'Injection & Infusion', 'Wound Care', 'General Supplies']
Length: 4, dtype: str
Weeks covered: 1 to 12


In [6]:
# Step 1: Calculate average weekly usage per branch per product
avg_usage = usage.groupby(['Branch', 'SKU'])['Units_Used'].mean().round(2).reset_index()
avg_usage.columns = ['Branch', 'SKU', 'Avg_Weekly_Usage']

# Step 2: Merge stock with avg usage
df = stock.merge(avg_usage, on=['Branch', 'SKU'], how='left')

# Step 3: Feature Engineering
# Weeks of supply remaining at current usage rate
df['Weeks_of_Supply'] = (df['Current_Stock'] / df['Avg_Weekly_Usage']).round(1)

# How far current stock is from reorder point (%)
df['Variance_Pct'] = ((df['Current_Stock'] - df['Reorder_Point']) / df['Reorder_Point'] * 100).round(1)

# Stock status flag
def classify_status(row):
    if row['Current_Stock'] <= row['Reorder_Point'] * 0.5:
        return 'Critical'
    elif row['Current_Stock'] < row['Reorder_Point']:
        return 'Understocked'
    elif row['Current_Stock'] > row['Max_Stock'] * 0.85:
        return 'Overstocked'
    else:
        return 'Healthy'

df['Stock_Status'] = df.apply(classify_status, axis=1)

# Reorder urgency based on weeks of supply
df['Reorder_Urgency'] = pd.cut(
    df['Weeks_of_Supply'],
    bins=[0, 2, 4, 8, float('inf')],
    labels=['Critical', 'High', 'Medium', 'Low']
)

print("=== FEATURE ENGINEERING COMPLETE ===")
print(df[['Branch','Product_Name','Current_Stock','Avg_Weekly_Usage',
          'Weeks_of_Supply','Variance_Pct','Stock_Status','Reorder_Urgency']].head(8).to_string())
print(f"\nStatus distribution:")
print(df['Stock_Status'].value_counts())

=== FEATURE ENGINEERING COMPLETE ===
         Branch                          Product_Name  Current_Stock  Avg_Weekly_Usage  Weeks_of_Supply  Variance_Pct  Stock_Status Reorder_Urgency
0  North Branch  Nitrile Examination Gloves (Box/100)            148            103.75              1.4         -26.0  Understocked        Critical
1  North Branch          Surgical Face Masks (Box/50)            576             79.33              7.3         284.0   Overstocked          Medium
2  North Branch              N95 Respirators (Box/20)             35             49.67              0.7         -65.0      Critical        Critical
3  North Branch            Isolation Gowns Disposable            393            149.92              2.6          31.0       Healthy            High
4  North Branch               Face Shields Disposable            125             48.83              2.6          25.0       Healthy            High
5  North Branch              Syringes 5ml with Needle            108       

In [7]:
# Save enriched dataset
df.to_csv('../data/inventory_analysis.csv', index=False)
print(f"Saved: {len(df)} rows, {len(df.columns)} columns")
print(f"Columns: {df.columns.tolist()}")

Saved: 80 rows, 13 columns
Columns: ['Branch', 'SKU', 'Product_Name', 'Category', 'Current_Stock', 'Reorder_Point', 'Max_Stock', 'Unit_Price_USD', 'Avg_Weekly_Usage', 'Weeks_of_Supply', 'Variance_Pct', 'Stock_Status', 'Reorder_Urgency']
